# Part 2 — Phase 3: Is the Forecasting Model Fair?

> A model that's accurate on average can still fail an entire group of residents.
> This notebook is where your team finds out **who the model fails — and decides
> whether to publish a benchmark anyway**.

**Your task:** A baseline forecasting model has already been run on the housing
dataset. Your job is to (1) measure overall accuracy, (2) test whether it's
*equally* accurate across property types, postcode areas, reference-record types,
resident counts, and seasons, and (3) write a Housing Benchmark Card stating
whether this dataset and model are ready for public release.

This is the convergence point — Tracks A, B, and C come together to produce one
shared output: `reference/benchmark_card.json`.

**Time budget:** ~2 hours.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import json
import re

os.makedirs('../reference', exist_ok=True)

df = pd.read_csv('../data/raw/housing_with_forecasts.csv',
                 parse_dates=['timestamp_recorded','timestamp_actual'])

print(f'Loaded: {df.shape}')
print(f'Households: {df.household_id.nunique()}')
print(f'Property types: {dict(df.drop_duplicates("household_id").property_type.value_counts())}')
df.head(3)

## Step 1 — Overall forecast performance (~20 min)

How well does the baseline model perform on average?

We compute three standard metrics on the holdout-equivalent rows:
- **MAE** (Mean Absolute Error): the typical size of an error in kWh
- **RMSE** (Root Mean Squared Error): penalises large errors more heavily
- **MBE** (Mean Bias Error): is the model systematically over- or under-forecasting?

A low MAE/RMSE means the model is accurate. A non-zero MBE means it has a
**systematic bias** — and that's where equity problems start.

In [ ]:
df_clean = df.dropna(subset=['forecast_model','smart_meter_kwh']).copy()
df_clean['abs_error']    = df_clean['model_error'].abs()
df_clean['squared_error'] = df_clean['model_error']**2

mae  = df_clean['abs_error'].mean()
rmse = np.sqrt(df_clean['squared_error'].mean())
mbe  = df_clean['model_error'].mean()

print(f'Overall MAE:   {mae:.3f} kWh')
print(f'Overall RMSE:  {rmse:.3f} kWh')
print(f'Overall MBE:   {mbe:+.3f} kWh   (positive = model UNDER-forecasts; negative = over-forecasts)')

print(f'\nNaive baseline MAE: {df_clean.naive_error.abs().mean():.3f} kWh   '
      f'(should be worse than model)')

## Step 2 — Equity breakdown by property type (~30 min)

Now the critical question: is the model equally accurate for flats, terraced
houses, and semi-detached houses?

If MBE for flats is significantly different from zero (and from other groups),
the model is **biased against flat residents** — it will under- or over-predict
their energy use systematically, which has real consequences if it's used to
flag fuel poverty risk.

We slice the same metrics by property type and visualise both:
- A bar chart of MBE per group (where is the bias largest?)
- A scatter plot of actual vs forecast (does one group sit off the diagonal?)

In [ ]:
# Equity breakdown by property type
metrics = df_clean.groupby('property_type').agg(
    MAE  = ('abs_error', 'mean'),
    RMSE = ('squared_error', lambda x: np.sqrt(x.mean())),
    MBE  = ('model_error', 'mean'),
    n_hours = ('model_error', 'count'),
).round(3)
print(metrics)

worst_group = metrics['MBE'].abs().idxmax()
print(f'\nWorst-biased group: {worst_group}  (MBE = {metrics.loc[worst_group, "MBE"]:+.3f} kWh)')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

metrics['MBE'].plot(kind='bar', ax=axes[0], color=['#3b82f6','#ef4444','#10b981'])
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_title('Mean Bias Error by property type')
axes[0].set_ylabel('MBE (kWh)  | positive = under-forecast')
axes[0].set_xlabel('')

for ptype, color in zip(['flat','terraced','semi-detached'], ['#3b82f6','#ef4444','#10b981']):
    sub = df_clean[df_clean.property_type == ptype].sample(min(2000, len(df_clean)), random_state=1)
    axes[1].scatter(sub['smart_meter_kwh'], sub['forecast_model'], alpha=0.2, s=8, label=ptype, color=color)

lims = [0, max(df_clean['smart_meter_kwh'].max(), df_clean['forecast_model'].max())]
axes[1].plot(lims, lims, 'k--', alpha=0.5, label='perfect prediction')
axes[1].set_xlabel('Actual kWh')
axes[1].set_ylabel('Forecast kWh')
axes[1].set_title('Actual vs forecast by property type')
axes[1].legend()

plt.tight_layout()
plt.show()

## Step 3 — Equity check by resident count and season (~25 min)

Property type isn't the only equity dimension. Two more groups often get
under-represented in training data and ignored in evaluation:

- **Resident count** — does the model work equally for single-occupant
  households and 4-person families? (n_residents in the dataset.)
- **Season** — energy use patterns shift seasonally. The forecasts file ships
  with only October data, so the by-month cell below will return a single
  row; treat it as a stub and, if you regenerate the forecasts over the full
  October→December window in the main file, the seasonal signal becomes
  visible here.

You don't need to be comprehensive — pick the dimension that seems most likely
to reveal a problem and dig in.

In [ ]:
# Equity by resident count
print('=== By resident count ===')
print(df_clean.groupby('n_residents').agg(
    MAE  = ('abs_error', 'mean'),
    MBE  = ('model_error', 'mean'),
    n    = ('model_error', 'count'),
).round(3))

# By month (covers the 3-month span in the dataset)
df_clean['month'] = df_clean['timestamp_actual'].dt.month
print('\n=== By month ===')
print(df_clean.groupby('month').agg(
    MAE  = ('abs_error', 'mean'),
    MBE  = ('model_error', 'mean'),
).round(3))

## Step 4 — Equity by postcode area and reference type (~20 min)

Two more axes that often surface bias hidden by the property-type slice:

**Postcode area** — group properties by the first letter cluster of their
`postcode` (e.g. `SW`, `NW`, `CR`). Geographic clustering catches biases the
property-type slice misses.

**Reference type** — split properties by whether their `reference` is a
structured ID (`U######`) or a free-text address. This is often a proxy
for property registration era or council subsystem; if one side has worse
errors, it can be a sign the model is missing whole cohorts of properties.

In [ ]:
props = df_clean.drop_duplicates('household_id').set_index('household_id')[
    ['property_type','reference','Sub-building','address','postcode']
]
props['postcode_area'] = props['postcode'].astype(str).str.extract(r'^([A-Z]+)', expand=False)
props['reference_type'] = props['reference'].apply(
    lambda s: 'structured' if re.match(r'^U\d{6}$', str(s)) else 'free_text'
)

df_clean = df_clean.merge(
    props[['postcode_area','reference_type']],
    left_on='household_id', right_index=True
)

print('=== By postcode area ===')
print(df_clean.groupby('postcode_area').agg(
    MAE  = ('abs_error', 'mean'),
    MBE  = ('model_error', 'mean'),
    n_households = ('household_id', 'nunique'),
).sort_values('MBE', ascending=False).round(3))

print('\n=== By reference type (structured vs free-text) ===')
print(df_clean.groupby('reference_type').agg(
    MAE = ('abs_error', 'mean'),
    MBE = ('model_error', 'mean'),
    n_households = ('household_id', 'nunique'),
).round(3))

## Step 5 — Coverage equity (~15 min)

Performance equity is one half. The other half — and often the louder signal — is
**coverage equity**: whether the *data itself* is equally observed across groups.

`avgCo2` has significant missingness (~24% overall) including multi-day outages
for some properties. If those gaps fall mostly on flats, or mostly on properties
with free-text references, then a model that depends on CO₂ will be quietly
under-serving those groups even when overall accuracy looks fine.

This step quantifies it.

In [ ]:
avgco2_missing = df.merge(
    props[['postcode_area','reference_type']],
    left_on='household_id', right_index=True
)
avgco2_missing['avgCo2_missing'] = avgco2_missing['avgCo2'].isna()

print('avgCo2 missing fraction by property_type:')
print(avgco2_missing.groupby('property_type')['avgCo2_missing'].mean().round(3))

print('\navgCo2 missing fraction by reference_type:')
print(avgco2_missing.groupby('reference_type')['avgCo2_missing'].mean().round(3))

print('\navgCo2 missing fraction by postcode area (top 5):')
print(avgco2_missing.groupby('postcode_area')['avgCo2_missing'].mean()
      .sort_values(ascending=False).head().round(3))

CONCERN = 0.25
print(f'\nGroups exceeding {CONCERN:.0%} avgCo2 missingness are flagged as coverage-equity concerns.')

## Step 6 — Is the flat bias statistically significant? (~20 min)

A bias chart isn't enough on its own — judges and reviewers will ask whether
the gap could be explained by chance. We run a t-test comparing the model
errors for flats against the model errors for terraced houses.

**If p < 0.05**, the bias is statistically significant — meaning it's unlikely
to be random noise, and the model has a real fairness problem you need to
flag in the benchmark card.

In [ ]:
errors_flat     = df_clean[df_clean.property_type=='flat']['model_error'].dropna()
errors_terraced = df_clean[df_clean.property_type=='terraced']['model_error'].dropna()

t_stat, p_value = stats.ttest_ind(errors_flat, errors_terraced, equal_var=False)
print(f't-statistic: {t_stat:.3f}')
print(f'p-value:     {p_value:.6f}')
print(f'\nMean error (flat):     {errors_flat.mean():+.3f} kWh')
print(f'Mean error (terraced): {errors_terraced.mean():+.3f} kWh')
print(f'\nDifference is statistically significant: {p_value < 0.05}')

## Step 7 — Build the Housing Benchmark Card (~45 min)

The Housing Benchmark Card is your team's headline output. It is a structured
JSON file that summarises:

- The dataset (what's in it, where its flaws are)
- The model (what was trained, on what splits)
- The equity report (who it fails, and by how much)
- Coverage equity (whose data the dataset under-observes)
- Known limitations and recommended uses

Save it to `reference/benchmark_card.json`. Judges check that file first.

In [ ]:
benchmark_card = {
    'schema': 'omaib-housing-v0.1',
    'card_name': 'Smart Social Housing Forecasting Benchmark',
    'version': '0.1.0-hackathon',
    'authors': ['<your team name here>'],
    'dataset': {
        'name': 'Smart Social Housing Synthetic — Hackathon Starter',
        'n_households': int(df.household_id.nunique()),
        'n_rows_total': int(len(df)),
        'modalities': ['smart_meter_kwh','indoor_temp_c','co2_ppm','noise_db',
                       'survey_thermal_comfort','avgTemperature','avgHumidity','avgCo2',
                       'reference','Sub-building','address','postcode'],
        'known_flaws_quantified': {
            'avgCo2_missing_fraction': round(float(df['avgCo2'].isna().mean()), 4),
            'sub_building_blank_pct': round(float(
                (props['Sub-building'].fillna('').astype(str).str.strip() == '').sum() / len(props)
            ), 4),
        },
    },
    'model': {
        'type': 'household-type mean baseline',
        'overall_MAE_kwh':  round(float(mae), 3),
        'overall_RMSE_kwh': round(float(rmse), 3),
        'overall_MBE_kwh':  round(float(mbe), 3),
    },
    'equity_report': {
        'dimensions_tested': ['property_type','n_residents','month','postcode_area','reference_type'],
        'worst_group':         worst_group,
        'worst_group_MBE_kwh': round(float(metrics.loc[worst_group,'MBE']), 3),
        'statistically_significant': bool(p_value < 0.05),
        'p_value': max(float(p_value), 1e-300),
        't_statistic': round(float(t_stat), 3),
        'extra_findings': {
            'postcode_area':  'TODO: worst area + MBE from Step 4',
            'reference_type': 'TODO: structured vs free-text MBE from Step 4',
        },
    },
    'coverage_equity': {
        'metric': 'avgCo2_missing_fraction',
        'overall': round(float(df['avgCo2'].isna().mean()), 4),
        'by_property_type': {},
        'by_reference_type': {},
        'concern_threshold': 0.25,
        'notes': 'Groups exceeding the threshold are at higher risk of being under-modelled '
                 'because their environmental signal is sparse.',
    },
    'known_limitations': [
        'Survey response rate is only 2% — thermal comfort cannot be used as a fairness signal alone',
        'Timestamp drift affects ~30% of households',
        '120 households is small — confidence intervals on equity findings are wide',
        'reference column mixes structured IDs and free-text — normalise before joining external records',
        'Sub-building is blank for a meaningful fraction of properties',
        'avgCo2 has multi-day outages on some properties — confirm before using it as a model input',
    ],
    'verdict': 'CONDITIONAL',
    'recommended_uses': [
        'Use ONLY with stratified evaluation by property_type, postcode_area, and reference_type',
        'Flag the under-forecast bias for flats in any downstream policy use',
        'Re-train with balanced property_type and postcode_area sampling',
    ],
    'do_not_use_for': [
        'Direct fuel-poverty risk scoring without group-stratified validation',
        'Real housing eligibility decisions',
        'Any application without a coverage-equity audit of avgCo2',
    ],
}

card_path = '../reference/benchmark_card.json'
with open(card_path, 'w') as f:
    json.dump(benchmark_card, f, indent=2)
print(f'Saved Housing Benchmark Card → {card_path}')
print(f'Card has {len(benchmark_card)} top-level fields.')

## Step 8 — Write the equity statement (~20 min)

The benchmark card is the technical artefact. The **equity statement** is its
plain-English companion — three short paragraphs a housing policy officer
could read in two minutes.

This is a strong place to use an AI assistant. Feed it your worst-group MBE,
the postcode/reference-type findings, the coverage-equity numbers, and ask
for a draft. Then edit it for tone and accuracy.

Save it to `reference/equity_statement.md`.

**Required content:**
1. **Who is most affected?** Which group of residents would be hurt if a model
   like this were deployed today?
2. **How big is the gap?** Quantify the bias with numbers, not adjectives.
3. **What should happen before deployment?** Recommend the next concrete step
   (re-train, gather more data, deploy with stratified monitoring, etc.).

In [ ]:
equity_statement = """# Equity Statement: Smart Social Housing Forecasting Model

## Who is most affected
[Fill in: which group of residents would be hurt if this model were deployed?]

## How big is the gap
[Fill in: quantify the bias with numbers — MBE, % difference, p-value]

## What should happen next
[Fill in: re-train with balanced sampling? gather more data? deploy with caveats?]
"""

with open('../reference/equity_statement.md', 'w') as f:
    f.write(equity_statement)
print('Saved equity statement template → ../reference/equity_statement.md')
print('\nEdit the file to fill in your team\'s findings before submission.')